# Data collection - Paths

In [1]:
from pathlib import Path
import os

print("Current working directory:", os.getcwd())

raw_path = Path("Data_Opgave_3/995,000_rows.csv")
splits_dir = Path("Data_Opgave_3/splits")
processed_dir = Path("Data_Opgave_3/processed")
liar_dir = Path(r"Data_Opgave_3\LIAR")

processed_dir.mkdir(parents=True, exist_ok=True)

print("raw_path =", raw_path.resolve())
print("splits_dir =", splits_dir.resolve())
print("processed_dir =", processed_dir.resolve())

Current working directory: c:\Users\Bruger\VisualCode\GDS_Eksamensprojekt\GDS-Eksamen-FakeNews
raw_path = C:\Users\Bruger\VisualCode\GDS_Eksamensprojekt\GDS-Eksamen-FakeNews\Data_Opgave_3\995,000_rows.csv
splits_dir = C:\Users\Bruger\VisualCode\GDS_Eksamensprojekt\GDS-Eksamen-FakeNews\Data_Opgave_3\splits
processed_dir = C:\Users\Bruger\VisualCode\GDS_Eksamensprojekt\GDS-Eksamen-FakeNews\Data_Opgave_3\processed


# Data splitting

In [2]:
import csv
import random
from collections import Counter

csv.field_size_limit(2**31 - 1)

def compute_targets(label_counts: Counter, train=0.8, val=0.1, test=0.1):
    targets = {}
    for label, n in label_counts.items():
        n_train = int(round(n * train))
        n_val = int(round(n * val))
        n_test = n - n_train - n_val
        targets[label] = {"train": n_train, "val": n_val, "test": n_test}
    return targets

def choose_split(remaining: dict, rng: random.Random):
    total = remaining["train"] + remaining["val"] + remaining["test"]
    if total <= 0:
        return None
    r = rng.randrange(total)
    if r < remaining["train"]:
        return "train"
    r -= remaining["train"]
    if r < remaining["val"]:
        return "val"
    return "test"

in_path = raw_path
seed = 42

splits_dir.mkdir(parents=True, exist_ok=True)

train_path = splits_dir / "train.csv"
val_path = splits_dir / "val.csv"
test_path = splits_dir / "test.csv"

label_counts = Counter()
with open(in_path, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        label_counts[row["type"]] += 1

targets = compute_targets(label_counts, train=0.8, val=0.1, test=0.1)
rng = random.Random(seed)
written = Counter()

with open(in_path, "r", encoding="utf-8", newline="") as f_in, \
     open(train_path, "w", encoding="utf-8", newline="") as f_train, \
     open(val_path, "w", encoding="utf-8", newline="") as f_val, \
     open(test_path, "w", encoding="utf-8", newline="") as f_test:

    reader = csv.DictReader(f_in)
    fieldnames = reader.fieldnames

    w_train = csv.DictWriter(f_train, fieldnames=fieldnames)
    w_val = csv.DictWriter(f_val, fieldnames=fieldnames)
    w_test = csv.DictWriter(f_test, fieldnames=fieldnames)

    w_train.writeheader()
    w_val.writeheader()
    w_test.writeheader()

    for row in reader:
        label = row["type"]
        split = choose_split(targets[label], rng)

        if split == "train":
            w_train.writerow(row)
        elif split == "val":
            w_val.writerow(row)
        else:
            w_test.writerow(row)

        targets[label][split] -= 1
        written[split] += 1

print("Wrote:", dict(written))
print("train exists?", train_path.exists())
print("val exists?", val_path.exists())
print("test exists?", test_path.exists())

Wrote: {'val': 99498, 'train': 796000, 'test': 99502}
train exists? True
val exists? True
test exists? True


# Data Preprocessing

In [3]:
import os
import re
import csv
import nltk
from pathlib import Path
from joblib import Parallel, delayed
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

csv.field_size_limit(2**31 - 1)

MONTH_DATE_RE = re.compile(
    r"\b(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)"
    r"(?!\.\w)"
    r"(?:\.(?=\s|$|\d))?"
    r"(?:\s+(?:0?[1-9]|[12][0-9]|3[01])"
    r"(?:\s*,?\s*(?:\d{2}|\d{4}))?"
    r")?"
    r"(?=\W|$)"
)

NUMBER_RE = re.compile(r"\b\d+(?:[.,]\d+)?\b")
URL_HTTP_RE = re.compile(r"https?://\S+")
URL_WWW_RE = re.compile(r"www\.\S+")
URL_TLD_RE = re.compile(r"\S+\.(?:com|org|net|io|gov|edu)\S*", re.IGNORECASE)
WHITESPACE_RE = re.compile(r"\s+")

TOKEN_RE = re.compile(r"url_token|num_token|date_token|[a-z]+(?:'[a-z]+)?")
PLACEHOLDER_TOKENS = {"url_token", "num_token", "date_token"}

def clean_text(text):
    text = str(text).lower()

    text = MONTH_DATE_RE.sub("date_token", text)
    text = NUMBER_RE.sub("num_token", text)

    text = URL_HTTP_RE.sub("url_token", text)
    text = URL_WWW_RE.sub("url_token", text)
    text = URL_TLD_RE.sub("url_token", text)

    text = WHITESPACE_RE.sub(" ", text).strip()
    return text


def tokenize_nltk(cleaned_text):
    return TOKEN_RE.findall(cleaned_text)


try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))

NEGATIONS = {"no", "not", "nor", "never", "n't"}
print("Negations removed by STOPWORDS:", NEGATIONS & STOPWORDS)

STOPWORDS = STOPWORDS - NEGATIONS


def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS]


stemmer = SnowballStemmer("english")


def stem_tokens(tokens):
    out = []
    for t in tokens:
        if t in PLACEHOLDER_TOKENS:
            out.append(t)
        else:
            out.append(stemmer.stem(t))
    return out


def reduction_rate(before, after):
    if before == 0:
        return 0.0
    return (before - after) / before


def _process_chunk(rows, use_stemming=False):
    vocab_raw = set()
    vocab_no_stop = set()
    vocab_final = set()

    total_raw = 0
    total_no_stop = 0

    out_rows = []

    for article_type, raw_content in rows:
        cleaned = clean_text(raw_content)

        tokens = tokenize_nltk(cleaned)
        tokens_no_stop = remove_stopwords(tokens)

        if use_stemming:
            tokens_final = stem_tokens(tokens_no_stop)
        else:
            tokens_final = tokens_no_stop

        total_raw += len(tokens)
        total_no_stop += len(tokens_no_stop)

        vocab_raw.update(tokens)
        vocab_no_stop.update(tokens_no_stop)
        vocab_final.update(tokens_final)

        out_rows.append((article_type, " ".join(tokens_final)))

    return {
        "rows": out_rows,
        "vocab_raw": vocab_raw,
        "vocab_no_stop": vocab_no_stop,
        "vocab_final": vocab_final,
        "total_raw": total_raw,
        "total_no_stop": total_no_stop,
        "docs_read": len(rows),
    }


def _iter_row_chunks(reader, chunk_size=2000, limit_rows=None):
    chunk = []
    docs_seen = 0

    for row in reader:
        if limit_rows is not None and docs_seen >= limit_rows:
            break

        article_type = row.get("type")
        raw_content = row.get("content", "") or ""

        chunk.append((article_type, raw_content))
        docs_seen += 1

        if len(chunk) >= chunk_size:
            yield chunk
            chunk = []

    if chunk:
        yield chunk


def _run_parallel_batch(batch_chunks, use_stemming, n_jobs, verbose):
    return Parallel(
        n_jobs=n_jobs,
        backend="loky",
        verbose=verbose
    )(
        delayed(_process_chunk)(chunk, use_stemming)
        for chunk in batch_chunks
    )


def process_file(
    file_path,
    out_path,
    limit_rows=None,
    use_stemming=False,
    n_jobs=None,
    chunk_size=2000,
    chunks_per_batch=None,
    verbose=0
):
    file_path = Path(file_path)
    out_path = Path(out_path)

    cpu_count = os.cpu_count() or 1

    if n_jobs is None:
        n_jobs = max(1, cpu_count - 1)
    elif n_jobs == -1:
        n_jobs = cpu_count

    if chunks_per_batch is None:
        chunks_per_batch = max(2 * n_jobs, 8)

    vocab_raw = set()
    vocab_no_stop = set()
    vocab_final = set()

    total_raw = 0
    total_no_stop = 0
    docs_read = 0

    with open(file_path, "r", encoding="utf-8-sig", newline="") as f_in, \
         open(out_path, "w", encoding="utf-8", newline="") as f_out:

        reader = csv.DictReader(f_in)
        writer = csv.DictWriter(f_out, fieldnames=["type", "processed_content"])
        writer.writeheader()

        batch_chunks = []

        for chunk in _iter_row_chunks(reader, chunk_size=chunk_size, limit_rows=limit_rows):
            batch_chunks.append(chunk)

            if len(batch_chunks) >= chunks_per_batch:
                results = _run_parallel_batch(
                    batch_chunks=batch_chunks,
                    use_stemming=use_stemming,
                    n_jobs=n_jobs,
                    verbose=verbose
                )

                for res in results:
                    docs_read += res["docs_read"]
                    total_raw += res["total_raw"]
                    total_no_stop += res["total_no_stop"]

                    vocab_raw.update(res["vocab_raw"])
                    vocab_no_stop.update(res["vocab_no_stop"])
                    vocab_final.update(res["vocab_final"])

                    for article_type, processed_content in res["rows"]:
                        writer.writerow({
                            "type": article_type,
                            "processed_content": processed_content
                        })

                batch_chunks = []

        if batch_chunks:
            results = _run_parallel_batch(
                batch_chunks=batch_chunks,
                use_stemming=use_stemming,
                n_jobs=n_jobs,
                verbose=verbose
            )

            for res in results:
                docs_read += res["docs_read"]
                total_raw += res["total_raw"]
                total_no_stop += res["total_no_stop"]

                vocab_raw.update(res["vocab_raw"])
                vocab_no_stop.update(res["vocab_no_stop"])
                vocab_final.update(res["vocab_final"])

                for article_type, processed_content in res["rows"]:
                    writer.writerow({
                        "type": article_type,
                        "processed_content": processed_content
                    })

    V_raw = len(vocab_raw)
    V_no_stop = len(vocab_no_stop)
    V_final = len(vocab_final)

    rr_stop = reduction_rate(V_raw, V_no_stop)
    rr_final = reduction_rate(V_no_stop, V_final)
    rr_final_from_raw = reduction_rate(V_raw, V_final)
    token_rr_stop = (total_raw - total_no_stop) / total_raw if total_raw else 0

    print(f"\nProcessed file: {file_path.name}")
    print(f"Saved to: {out_path.name}")
    print(f"Sample size (documents): {docs_read:,}")
    print(f"Vocabulary size (tokenized): {V_raw:,}")
    print(f"Vocabulary size (after stopwords): {V_no_stop:,}")
    print(f"Reduction rate (stopwords): {rr_stop:.4f} = {rr_stop*100:.2f}%")
    print(f"Vocabulary size (final): {V_final:,}")
    print(f"Reduction rate (final vs no_stop): {rr_final:.4f} = {rr_final*100:.2f}%")
    print(f"Reduction rate (final vs raw): {rr_final_from_raw:.4f} = {rr_final_from_raw*100:.2f}%")
    print(f"Total tokens (raw): {total_raw:,}")
    print(f"Total tokens (after stopwords): {total_no_stop:,}")
    print(f"Token reduction (stopwords): {token_rr_stop:.4f} = {token_rr_stop*100:.2f}%")
    print(f"Stemming enabled: {use_stemming}")
    print(f"n_jobs used: {n_jobs}")
    print(f"chunk_size: {chunk_size}")
    print(f"chunks_per_batch: {chunks_per_batch}")


n_jobs = None
chunk_size = 10000
chunks_per_batch = 16

raw_paths = {
    "train": splits_dir / "train.csv",
    "val": splits_dir / "val.csv",
    "test": splits_dir / "test.csv"
}

processed_paths = {
    "train": processed_dir / "train_processed.csv",
    "val": processed_dir / "val_processed.csv",
    "test": processed_dir / "test_processed.csv"
}

print("=== Inputfiler før preprocessing ===")
for split, path in raw_paths.items():
    print(f"{split}: {path.exists()} -> {path}")

for split in ["train", "val", "test"]:
    input_path = raw_paths[split]
    output_path = processed_paths[split]

    if not input_path.exists():
        raise FileNotFoundError(f"Mangler inputfil: {input_path}")

    print(f"\nKører preprocessing for {split}...")
    process_file(
        file_path=input_path,
        out_path=output_path,
        limit_rows=None,
        use_stemming=False,
        n_jobs=n_jobs,
        chunk_size=chunk_size,
        chunks_per_batch=chunks_per_batch,
        verbose=0
    )

print("\n=== Inputfiler efter preprocessing ===")
for split, path in raw_paths.items():
    print(f"{split}: {path.exists()} -> {path}")

print("\n=== Outputfiler efter preprocessing ===")
for split, path in processed_paths.items():
    print(f"{split}: {path.exists()} -> {path}")

Negations removed by STOPWORDS: {'nor', 'no', 'not'}
=== Inputfiler før preprocessing ===
train: True -> Data_Opgave_3\splits\train.csv
val: True -> Data_Opgave_3\splits\val.csv
test: True -> Data_Opgave_3\splits\test.csv

Kører preprocessing for train...

Processed file: train.csv
Saved to: train_processed.csv
Sample size (documents): 796,000
Vocabulary size (tokenized): 884,877
Vocabulary size (after stopwords): 884,682
Reduction rate (stopwords): 0.0002 = 0.02%
Vocabulary size (final): 884,682
Reduction rate (final vs no_stop): 0.0000 = 0.00%
Reduction rate (final vs raw): 0.0002 = 0.02%
Total tokens (raw): 374,864,882
Total tokens (after stopwords): 218,178,919
Token reduction (stopwords): 0.4180 = 41.80%
Stemming enabled: False
n_jobs used: 7
chunk_size: 10000
chunks_per_batch: 16

Kører preprocessing for val...

Processed file: val.csv
Saved to: val_processed.csv
Sample size (documents): 99,498
Vocabulary size (tokenized): 291,864
Vocabulary size (after stopwords): 291,670
Reduct

# Feature representation TF - IDF

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

train = pd.read_csv(processed_dir / "train_processed.csv")
val = pd.read_csv(processed_dir / "val_processed.csv")
test = pd.read_csv(processed_dir / "test_processed.csv")

print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())

labels = {
    "reliable": "real",
    "political": "real",
    "fake": "fake",
    "unreliable": "fake",
    "conspiracy": "fake",
    "bias": "fake",
    "hate": "fake",
    "clickbait": "fake"
}

for df in [train, val, test]:
    # Lav binær label ud fra den originale type-kolonne
    df["binary_label"] = df["type"].map(labels)

    # Fjern rækker med manglende tekst eller labels
    df.dropna(subset=["binary_label", "processed_content"], inplace=True)

    # Sørg for at teksten er string
    df["processed_content"] = df["processed_content"].astype(str)

    # Fjern tomme tekster
    df.drop(df[df["processed_content"].str.strip() == ""].index, inplace=True)

print("\nLabel-fordeling efter mapping:")
print("Train:")
print(train["binary_label"].value_counts(dropna=False))
print("\nVal:")
print(val["binary_label"].value_counts(dropna=False))
print("\nTest:")
print(test["binary_label"].value_counts(dropna=False))

label_to_int = {
    "fake": 0,
    "real": 1
}

y_train = train["binary_label"].map(label_to_int).astype(np.int8)
y_val = val["binary_label"].map(label_to_int).astype(np.int8)
y_test = test["binary_label"].map(label_to_int).astype(np.int8)

print("\nNumerisk label-fordeling:")
print("y_train:")
print(y_train.value_counts(dropna=False))
print("\ny_val:")
print(y_val.value_counts(dropna=False))
print("\ny_test:")
print(y_test.value_counts(dropna=False))

if y_train.nunique() < 2:
    raise ValueError(f"y_train indeholder kun én klasse: {y_train.unique()}")

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    stop_words=None,
    max_features=200000,
    dtype=np.float32
)

X_train = vectorizer.fit_transform(train["processed_content"])
X_val = vectorizer.transform(val["processed_content"])
X_test = vectorizer.transform(test["processed_content"])

print("\nTF-IDF shapes:")
print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

idf_dir = Path("Data_Opgave_3/idf_values")
idf_dir.mkdir(parents=True, exist_ok=True)

features = vectorizer.get_feature_names_out()
idf_values = vectorizer.idf_

idf_df = pd.DataFrame({
    "term": features,
    "idf": idf_values
})

idf_output_path = idf_dir / "train_idf_values.csv"
idf_df.to_csv(idf_output_path, index=False, encoding="utf-8")

print("\nIDF-fil gemt her:", idf_output_path.resolve())
print(idf_df.head(20))

Train columns: ['type', 'processed_content']
Val columns: ['type', 'processed_content']
Test columns: ['type', 'processed_content']

Label-fordeling efter mapping:
Train:
binary_label
real    330463
fake    323971
Name: count, dtype: int64

Val:
binary_label
real    41308
fake    40500
Name: count, dtype: int64

Test:
binary_label
real    41309
fake    40498
Name: count, dtype: int64

Numerisk label-fordeling:
y_train:
binary_label
1    330463
0    323971
Name: count, dtype: int64

y_val:
binary_label
1    41308
0    40500
Name: count, dtype: int64

y_test:
binary_label
1    41309
0    40498
Name: count, dtype: int64

TF-IDF shapes:
Train: (654434, 200000)
Val:   (81808, 200000)
Test:  (81807, 200000)

IDF-fil gemt her: C:\Users\Bruger\VisualCode\GDS_Eksamensprojekt\GDS-Eksamen-FakeNews\Data_Opgave_3\idf_values\train_idf_values.csv
             term        idf
0              aa   7.737375
1    aa num_token  10.217140
2             aaa   7.491805
3      aaa rating   9.891718
4          

## Linear SVM

In [5]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

def evaluate_linear_svc_tfidf(X_train, X_eval, y_train, y_eval, base_dir, dataset_name):
    model_name = "LinearSVC_TFIDF"
    model_dir = base_dir / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    model = LinearSVC(
        C=0.3,
        class_weight="balanced",
        dual="auto",
        max_iter=5000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_eval)
    decision_scores = model.decision_function(X_eval)

    acc = accuracy_score(y_eval, y_pred)
    prec_fake = precision_score(y_eval, y_pred, pos_label=0, zero_division=0)
    rec_fake = recall_score(y_eval, y_pred, pos_label=0, zero_division=0)
    f1_fake = f1_score(y_eval, y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_eval, y_pred, average="macro")

    report = classification_report(
        y_eval,
        y_pred,
        labels=[0, 1],
        target_names=["fake", "real"],
        zero_division=0
    )

    cm = confusion_matrix(y_eval, y_pred, labels=[0, 1])

    print(f"\n=== Results on {dataset_name} - {model_name} ===")
    print("Accuracy:", acc)
    print("Precision_fake:", prec_fake)
    print("Recall_fake:", rec_fake)
    print("F1_fake:", f1_fake)
    print("F1_macro:", f1_macro)

    print("\nClassification report:")
    print(report)

    print("\nConfusion matrix:")
    print(cm)

    metrics_df = pd.DataFrame({
        "model": [model_name],
        "dataset": [dataset_name],
        "accuracy": [acc],
        "precision_fake": [prec_fake],
        "recall_fake": [rec_fake],
        "f1_fake": [f1_fake],
        "f1_macro": [f1_macro]
    })

    metrics_path = model_dir / f"{dataset_name}_metrics.csv"
    metrics_df.to_csv(metrics_path, index=False, encoding="utf-8")

    cm_df = pd.DataFrame(
        cm,
        index=["true_fake", "true_real"],
        columns=["pred_fake", "pred_real"]
    )

    cm_path = model_dir / f"confusion_matrix_{dataset_name}.csv"
    cm_df.to_csv(cm_path, encoding="utf-8")

    report_path = model_dir / f"classification_report_{dataset_name}.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)

    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["fake", "real"])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"{model_name} - {dataset_name}")
    fig.tight_layout()

    cm_png_path = model_dir / f"confusion_matrix_{dataset_name}.png"
    fig.savefig(cm_png_path, dpi=200)
    plt.close(fig)

    int_to_label = {0: "fake", 1: "real"}

    if list(model.classes_) == [0, 1]:
        fake_score = -decision_scores
    else:
        fake_score = decision_scores

    pred_df = pd.DataFrame({
        "true_label_num": pd.Series(y_eval).reset_index(drop=True),
        "predicted_label_num": pd.Series(y_pred).reset_index(drop=True),
        "true_label": pd.Series(y_eval).map(int_to_label).reset_index(drop=True),
        "predicted_label": pd.Series(y_pred).map(int_to_label).reset_index(drop=True),
        "decision_score_for_fake": pd.Series(fake_score).reset_index(drop=True)
    })

    predictions_path = model_dir / f"{dataset_name}_predictions.csv"
    pred_df.to_csv(predictions_path, index=False, encoding="utf-8")

    return {
        "model": model_name,
        "dataset": dataset_name,
        "accuracy": acc,
        "precision_fake": prec_fake,
        "recall_fake": rec_fake,
        "f1_fake": f1_fake,
        "f1_macro": f1_macro,
        "metrics_path": str(metrics_path.resolve()),
        "cm_path": str(cm_path.resolve()),
        "cm_png_path": str(cm_png_path.resolve()),
        "report_path": str(report_path.resolve()),
        "predictions_path": str(predictions_path.resolve())
    }


results_dir = Path("Data_Opgave_3/LinearSVM_results")
results_dir.mkdir(parents=True, exist_ok=True)

val_result = evaluate_linear_svc_tfidf(
    X_train=X_train,
    X_eval=X_val,
    y_train=y_train,
    y_eval=y_val,
    base_dir=results_dir,
    dataset_name="validation"
)

summary_df = pd.DataFrame([{
    "model": val_result["model"],
    "dataset": val_result["dataset"],
    "accuracy": val_result["accuracy"],
    "precision_fake": val_result["precision_fake"],
    "recall_fake": val_result["recall_fake"],
    "f1_fake": val_result["f1_fake"],
    "f1_macro": val_result["f1_macro"]
}])

summary_path = results_dir / "validation_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8")

print("\n=== Validation summary ===")
print(summary_df)



train_val = pd.concat([train, val], axis=0).reset_index(drop=True)
y_train_val = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

vectorizer_final = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    stop_words=None,
    max_features=200000,
    dtype=np.float32
)

X_train_val = vectorizer_final.fit_transform(train_val["processed_content"])
X_test_final = vectorizer_final.transform(test["processed_content"])

test_result = evaluate_linear_svc_tfidf(
    X_train=X_train_val,
    X_eval=X_test_final,
    y_train=y_train_val,
    y_eval=y_test,
    base_dir=results_dir,
    dataset_name="test"
)

test_summary_df = pd.DataFrame([{
    "model": test_result["model"],
    "dataset": test_result["dataset"],
    "accuracy": test_result["accuracy"],
    "precision_fake": test_result["precision_fake"],
    "recall_fake": test_result["recall_fake"],
    "f1_fake": test_result["f1_fake"],
    "f1_macro": test_result["f1_macro"]
}])

test_summary_path = results_dir / "test_summary.csv"
test_summary_df.to_csv(test_summary_path, index=False, encoding="utf-8")

print("\n=== Test summary ===")
print(test_summary_df)


all_summary_df = pd.DataFrame([
    {
        "model": val_result["model"],
        "dataset": val_result["dataset"],
        "accuracy": val_result["accuracy"],
        "precision_fake": val_result["precision_fake"],
        "recall_fake": val_result["recall_fake"],
        "f1_fake": val_result["f1_fake"],
        "f1_macro": val_result["f1_macro"]
    },
    {
        "model": test_result["model"],
        "dataset": test_result["dataset"],
        "accuracy": test_result["accuracy"],
        "precision_fake": test_result["precision_fake"],
        "recall_fake": test_result["recall_fake"],
        "f1_fake": test_result["f1_fake"],
        "f1_macro": test_result["f1_macro"]
    }
])

all_summary_path = results_dir / "all_summary.csv"
all_summary_df.to_csv(all_summary_path, index=False, encoding="utf-8")
print(all_summary_df)


=== Results on validation - LinearSVC_TFIDF ===
Accuracy: 0.9166096225308038
Precision_fake: 0.9160448683105203
Recall_fake: 0.9154567901234568
F1_fake: 0.9157507348037641
F1_macro: 0.9166009548755637

Classification report:
              precision    recall  f1-score   support

        fake       0.92      0.92      0.92     40500
        real       0.92      0.92      0.92     41308

    accuracy                           0.92     81808
   macro avg       0.92      0.92      0.92     81808
weighted avg       0.92      0.92      0.92     81808


Confusion matrix:
[[37076  3424]
 [ 3398 37910]]

=== Validation summary ===
             model     dataset  accuracy  precision_fake  recall_fake  \
0  LinearSVC_TFIDF  validation   0.91661        0.916045     0.915457   

    f1_fake  f1_macro  
0  0.915751  0.916601  

=== Results on test - LinearSVC_TFIDF ===
Accuracy: 0.9192734118107252
Precision_fake: 0.9187339395137379
Recall_fake: 0.918144105881772
F1_fake: 0.918438927998024
F1_macro:

# Deployment / usage

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
)

processed_dir = Path("Data_Opgave_3/processed")
liar_processed_dir = Path("Data_Opgave_3/LIAR/processed")
output_dir = Path("Data_Opgave_3/LinearSVM_comparison/CrossDomain_Final")
output_dir.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(processed_dir / "train_processed.csv")
val = pd.read_csv(processed_dir / "val_processed.csv")
test = pd.read_csv(processed_dir / "test_processed.csv")
liar_test = pd.read_csv(liar_processed_dir / "test_processed.csv")

# FakeNews: vi accepterer både "real" og "reliable" som positiv klasse
main_label_map = {
    "fake": 0,
    "real": 1,
    "reliable": 1
}

# LIAR raw labels -> binære labels
liar_raw_to_binary = {
    "true": "real",
    "mostly-true": "real",
    "false": "fake",
    "barely-true": "fake",
    "pants-fire": "fake"
}

binary_to_y = {
    "fake": 0,
    "real": 1,
    "reliable": 1
}

def find_existing_column(df, candidates, df_name):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Ingen af disse kolonner findes i {df_name}: {candidates}")

def prepare_main_split(df, df_name):
    text_col = find_existing_column(df, ["processed_content"], df_name)
    label_col = find_existing_column(df, ["binary_label", "type"], df_name)

    df = df.dropna(subset=[text_col, label_col]).copy()

    df[text_col] = df[text_col].astype(str)
    df = df[df[text_col].str.strip() != ""].copy()

    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df["y"] = df[label_col].map(main_label_map)

    df = df[df["y"].notna()].copy()
    df["y"] = df["y"].astype(int)

    return df, text_col, label_col

def prepare_liar_split(df, df_name):
    text_col = find_existing_column(df, ["text_proc", "processed_content", "text"], df_name)

    # Hvis binary_label allerede findes i LIAR processed-filen, bruger vi den direkte
    if "binary_label" in df.columns:
        label_col = "binary_label"
        df = df.dropna(subset=[text_col, label_col]).copy()

        df[text_col] = df[text_col].astype(str)
        df = df[df[text_col].str.strip() != ""].copy()

        df[label_col] = df[label_col].astype(str).str.strip().str.lower()
        df["y"] = df[label_col].map(binary_to_y)

        df = df[df["y"].notna()].copy()
        df["y"] = df["y"].astype(int)
        return df, text_col, label_col

    # Ellers mapper vi fra original LIAR label
    label_col = find_existing_column(df, ["label"], df_name)

    df = df.dropna(subset=[text_col, label_col]).copy()

    df[text_col] = df[text_col].astype(str)
    df = df[df[text_col].str.strip() != ""].copy()

    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df["binary_label"] = df[label_col].map(liar_raw_to_binary)

    df = df[df["binary_label"].notna()].copy()
    df["y"] = df["binary_label"].map(binary_to_y).astype(int)

    return df, text_col, label_col

def evaluate_and_save(name, y_true, y_pred, output_dir):
    acc = accuracy_score(y_true, y_pred)
    prec_fake = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    rec_fake = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_fake = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["fake", "real"],
        zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"\n=== {name} ===")
    print("Accuracy:", acc)
    print("Precision_fake:", prec_fake)
    print("Recall_fake:", rec_fake)
    print("F1_fake:", f1_fake)
    print("F1_macro:", f1_macro)
    print("\nClassification report:")
    print(report)
    print("\nConfusion matrix:")
    print(cm)

    safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")

    # Save metrics
    metrics_df = pd.DataFrame([{
        "dataset": name,
        "accuracy": acc,
        "precision_fake": prec_fake,
        "recall_fake": rec_fake,
        "f1_fake": f1_fake,
        "f1_macro": f1_macro
    }])
    metrics_df.to_csv(output_dir / f"{safe_name}_metrics.csv", index=False, encoding="utf-8")

    # Save report
    with open(output_dir / f"{safe_name}_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report)

    # Save confusion matrix csv
    cm_df = pd.DataFrame(
        cm,
        index=["true_fake", "true_real"],
        columns=["pred_fake", "pred_real"]
    )
    cm_df.to_csv(output_dir / f"{safe_name}_confusion_matrix.csv", encoding="utf-8")

    # Save confusion matrix image
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["fake", "real"])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name)
    fig.tight_layout()
    fig.savefig(output_dir / f"{safe_name}_confusion_matrix.png", dpi=200)
    plt.close(fig)

    return {
        "Dataset": name,
        "Accuracy": acc,
        "Precision_fake": prec_fake,
        "Recall_fake": rec_fake,
        "F1_fake": f1_fake,
        "F1_macro": f1_macro
    }

train_eval, main_text_col, main_label_col = prepare_main_split(train, "train")
val_eval, _, _ = prepare_main_split(val, "val")
test_eval, _, _ = prepare_main_split(test, "test")
liar_test_bin, liar_text_col, liar_label_col = prepare_liar_split(liar_test, "liar_test")

print("FakeNews shapes:")
print("Train:", train_eval.shape)
print("Val:  ", val_eval.shape)
print("Test: ", test_eval.shape)

print("\nLIAR test shape:", liar_test_bin.shape)

if "binary_label" in liar_test_bin.columns:
    print(liar_test_bin["binary_label"].value_counts())
else:
    print(liar_test_bin["y"].value_counts())

trainval_eval = pd.concat([train_eval, val_eval], ignore_index=True)

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    stop_words=None,
    use_idf=True,
    norm="l2",
    max_features=200000,
    dtype=np.float32
)

model = LinearSVC(
    C=0.3,
    class_weight="balanced",
    dual="auto",
    max_iter=5000,
    random_state=42
)

X_trainval = vectorizer.fit_transform(trainval_eval[main_text_col])
y_trainval = trainval_eval["y"]

model.fit(X_trainval, y_trainval)

X_fake_test = vectorizer.transform(test_eval[main_text_col])
X_liar_test = vectorizer.transform(liar_test_bin[liar_text_col])

y_fake_test = test_eval["y"]
y_liar_test = liar_test_bin["y"]

y_fake_pred = model.predict(X_fake_test)
y_liar_pred = model.predict(X_liar_test)

fake_result = evaluate_and_save("FakeNews TEST", y_fake_test, y_fake_pred, output_dir)
liar_result = evaluate_and_save("LIAR TEST (no retraining)", y_liar_test, y_liar_pred, output_dir)

results = pd.DataFrame([
    {
        "Dataset": "FakeNews test",
        "Training_source": "FakeNews train+val",
        "Evaluation_type": "in-domain",
        "Accuracy": fake_result["Accuracy"],
        "Precision_fake": fake_result["Precision_fake"],
        "Recall_fake": fake_result["Recall_fake"],
        "F1_fake": fake_result["F1_fake"],
        "F1_macro": fake_result["F1_macro"],
    },
    {
        "Dataset": "LIAR test",
        "Training_source": "FakeNews train+val",
        "Evaluation_type": "cross-domain",
        "Accuracy": liar_result["Accuracy"],
        "Precision_fake": liar_result["Precision_fake"],
        "Recall_fake": liar_result["Recall_fake"],
        "F1_fake": liar_result["F1_fake"],
        "F1_macro": liar_result["F1_macro"],
    }
])

print("\n=== Final comparison ===")
print(results)

results.to_csv(output_dir / "final_results.csv", index=False, encoding="utf-8")

final_cfg = {
    "name": "LinearSVC_raw_TFIDF_final",
    "tfidf": {
        "analyzer": "word",
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True,
        "stop_words": None,
        "use_idf": True,
        "norm": "l2",
        "max_features": 200000,
        "dtype": "float32"
    },
    "model": {
        "type": "LinearSVC",
        "C": 0.3,
        "class_weight": "balanced",
        "dual": "auto",
        "max_iter": 5000,
        "random_state": 42
    },
    "training_setup": {
        "train_source": "FakeNews train+val",
        "evaluation_sets": ["FakeNews test", "LIAR test"],
        "chi_square": False,
        "svd": False
    }
}

joblib.dump(final_cfg, output_dir / "final_cfg.joblib")
joblib.dump(
    {
        "main_label_map": main_label_map,
        "liar_raw_to_binary": liar_raw_to_binary,
        "binary_to_y": binary_to_y
    },
    output_dir / "label_maps.joblib"
)
joblib.dump(
    {
        "main_text_col": main_text_col,
        "main_label_col": main_label_col,
        "liar_text_col": liar_text_col,
        "liar_label_col": liar_label_col
    },
    output_dir / "schema.joblib"
)
joblib.dump(vectorizer, output_dir / "final_vectorizer.joblib")
joblib.dump(model, output_dir / "final_model.joblib")

print("\nGemte filer i:")
print(output_dir.resolve())

FakeNews shapes:
Train: (258756, 3)
Val:   (32344, 3)
Test:  (32346, 3)

LIAR test shape: (1002, 5)
binary_label
fake    553
real    449
Name: count, dtype: int64

=== FakeNews TEST ===
Accuracy: 0.9774933531193966
Precision_fake: 0.9621247987879935
Recall_fake: 0.9687291448183811
F1_fake: 0.9654156769596199
F1_macro: 0.9743672491392665

Classification report:
              precision    recall  f1-score   support

        fake       0.96      0.97      0.97     10489
        real       0.98      0.98      0.98     21857

    accuracy                           0.98     32346
   macro avg       0.97      0.98      0.97     32346
weighted avg       0.98      0.98      0.98     32346


Confusion matrix:
[[10161   328]
 [  400 21457]]

=== LIAR TEST (no retraining) ===
Accuracy: 0.5339321357285429
Precision_fake: 0.5716666666666667
Recall_fake: 0.620253164556962
F1_fake: 0.5949696444058976
F1_macro: 0.523101743472044

Classification report:
              precision    recall  f1-score   supp

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

labels = ["fake", "real"]

# Test confusion matrix
cm = confusion_matrix(y_test, y_pred_test, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Confusion Matrix - LinearSVC on FakeNewsCorpus Test Set")
plt.tight_layout()
plt.show()

# Row-normalized version
cm_norm = confusion_matrix(y_test, y_pred_test, labels=[0, 1], normalize="true")

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", values_format=".2f")
plt.title("Normalized Confusion Matrix - LinearSVC on FakeNewsCorpus Test Set")
plt.tight_layout()
plt.show()